In [ ]:
import os, pickle, json
import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupShuffleSplit, StratifiedShuffleSplit
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, precision_recall_curve, confusion_matrix, roc_curve
)

# ---------- 0) Paths ----------
KOI_PATH = "../data/koi.csv"
TOI_PATH = "../data/toi.csv"
OUT_DIR  = "file"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------- 1) Constants ----------
CLEAN_FEATURES = [
    "koi_period","koi_duration","koi_depth","koi_ror","koi_impact",
    "koi_model_snr","koi_num_transits",
    "koi_prad","koi_srad","koi_steff","koi_kepmag",
    "koi_period_err1","koi_duration_err1","koi_depth_err1",
]
LOG_COLS = ["koi_period","koi_duration","koi_depth","koi_prad","koi_srad","koi_kepmag"]

# ---------- 2) Helpers ----------
def to_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    all_nan = [c for c in df.columns if df[c].isna().all()]
    if all_nan:
        df = df.drop(columns=all_nan)
    return df

def safe_roc_auc(y_true, prob):
    u = np.unique(y_true)
    return float('nan') if len(u) < 2 else roc_auc_score(y_true, prob)

def summarize_from_prob(y_true, prob, thr=0.5):
    pred = (prob>=thr).astype(int)
    return dict(
        Accuracy = accuracy_score(y_true, pred),
        Precision= precision_score(y_true, pred, zero_division=0),
        Recall   = recall_score(y_true, pred, zero_division=0),
        F1       = f1_score(y_true, pred, zero_division=0),
        ROC_AUC  = safe_roc_auc(y_true, prob),
        CM       = confusion_matrix(y_true, pred),
    )

def tune_thr_for_precision(y_true, prob, target=0.95):
    p, r, t = precision_recall_curve(y_true, prob)
    best = {"Thr":0.5, "Precision@tuned":0.0, "Recall@tuned":0.0}
    for P, R, T in zip(p[:-1], r[:-1], t):
        if P >= target and R > best["Recall@tuned"]:
            best = {"Thr": float(T), "Precision@tuned": float(P), "Recall@tuned": float(R)}
    return best

def best_thr_youden(y_true, prob):
    fpr, tpr, thr = roc_curve(y_true, prob)
    j = tpr - fpr
    return float(thr[int(np.argmax(j))])

# ---------- 3) Load KOI + labels (label = koi_pdisposition; fallback: koi_disposition) ----------
df_koi = pd.read_csv(KOI_PATH, comment="#", engine="python", on_bad_lines="skip")

if "koi_pdisposition" in df_koi.columns:
    yk_raw = df_koi["koi_pdisposition"].astype(str).str.upper().str.strip()
    mask_k = yk_raw.isin(["CONFIRMED","FALSE POSITIVE"])
    work_koi = df_koi.loc[mask_k].copy()
    work_koi["label"] = (yk_raw.loc[mask_k] == "CONFIRMED").astype(int)
elif "koi_disposition" in df_koi.columns:
    yk_raw = df_koi["koi_disposition"].astype(str).str.upper().str.strip()
    mask_k = yk_raw.isin(["CONFIRMED","FALSE POSITIVE"])
    work_koi = df_koi.loc[mask_k].copy()
    work_koi["label"] = (yk_raw.loc[mask_k] == "CONFIRMED").astype(int)
else:
    raise ValueError("KOI: label column not found (koi_pdisposition/koi_disposition).")

Xk = work_koi[[c for c in CLEAN_FEATURES if c in work_koi.columns]].copy()
Xk = to_numeric_df(Xk)
yk = work_koi["label"].astype(int).values
gk = work_koi["kepid"].astype(str).values
Xk["mission"] = "KOI"

# ---------- 4) Load TOI + labels + explicit feature mapping ----------
df_toi = pd.read_csv(TOI_PATH, comment="#", engine="python", on_bad_lines="skip")

assert "tfopwg_disp" in df_toi.columns, "TOI: tfopwg_disp not found."
yt_raw = df_toi["tfopwg_disp"].astype(str).str.upper().str.strip()
y_map_toi = {"CONFIRMED":1, "KNOWN PLANET":1, "KP":1, "CP":1, "FALSE POSITIVE":0, "FP":0}
mask_t = yt_raw.map(lambda v: v in y_map_toi)
work_toi = df_toi.loc[mask_t].copy()
yt = yt_raw.loc[mask_t].map(y_map_toi).astype(int).values

# TIC/TID group id
tic_col = None
for key in ["tid","tic","tic_id","ticid"]:
    for c in work_toi.columns:
        if key in c.lower():
            tic_col = c; break
    if tic_col: break
if tic_col is None:
    raise ValueError("TOI: TIC/TID column not found.")
gt = work_toi[tic_col].astype(str).values

# TOI -> KOI mapping (explicit)
toi_map = {
    "koi_period":        "pl_orbper",
    "koi_duration":      "pl_trandurh",
    "koi_depth":         "pl_trandep",
    "koi_prad":          "pl_rade",
    "koi_steff":         "st_teff",
    "koi_srad":          "st_rad",
    "koi_kepmag":        "st_tmag",
    "koi_period_err1":   "pl_orbpererr1",
    "koi_duration_err1": "pl_trandurherr1",
    "koi_depth_err1":    "pl_trandeperr1",
}
use_toi_feats = [k for k,v in toi_map.items() if v in work_toi.columns and k in CLEAN_FEATURES]
Xt_raw = work_toi[[toi_map[k] for k in use_toi_feats]].copy()
Xt_raw.columns = use_toi_feats
for c in Xt_raw.columns:
    Xt_raw[c] = pd.to_numeric(Xt_raw[c], errors="coerce")
Xt = Xt_raw.drop(columns=[c for c in Xt_raw.columns if Xt_raw[c].isna().all()]).copy()
Xt["mission"] = "TOI"

# ---------- 5) Align, concat, log-features ----------
common_feats = sorted(list(set(Xk.columns) & set(Xt.columns) & set(CLEAN_FEATURES)))
D = pd.concat([
    Xk[common_feats].assign(_y=yk, _gid=["KPL:"+g for g in gk]),
    Xt[common_feats].assign(_y=yt, _gid=["TESS:"+g for g in gt]),
], axis=0, ignore_index=True)

for c in common_feats:
    D[c] = pd.to_numeric(D[c], errors="coerce")

# add log features (and append to common_feats)
for col in LOG_COLS:
    if col in common_feats and f"log_{col}" not in common_feats:
        D[f"log_{col}"] = np.log1p(D[col].abs())
        common_feats.append(f"log_{col}")

y_all = D["_y"].astype(int).values
groups_all = D["_gid"].values

print(f"Common features ({len(common_feats)}):", common_feats)
print("Total rows:", len(D))
print("KOI class distribution:", np.bincount(yk))
print("TOI class distribution:", np.bincount(yt))

# ---------- 6) SPLIT: KOI -> TRAIN ONLY (KOI single-class), TEST from TOI  ----------
is_kpl_all = np.array([g.startswith("KPL") for g in groups_all])
is_tes_all = ~is_kpl_all

# 1) KOI: all Train
train_idx_koi = np.where(is_kpl_all)[0]
test_idx_koi  = np.array([], dtype=int)

# 2) TOI: GroupSplit with both classes in train/test
tes_idx_all = np.where(is_tes_all)[0]
X_tes_all   = D.loc[tes_idx_all, common_feats].values
y_tes_all   = y_all[tes_idx_all]
g_tes_all   = groups_all[tes_idx_all]

gss = GroupShuffleSplit(n_splits=60, test_size=0.15, random_state=42)
train_idx_tes = test_idx_tes = None
for i,(tr, te) in enumerate(gss.split(X_tes_all, y_tes_all, groups=g_tes_all)):
    if len(np.unique(y_tes_all[tr]))==2 and len(np.unique(y_tes_all[te]))==2:
        train_idx_tes = tes_idx_all[tr]
        test_idx_tes  = tes_idx_all[te]
        print(f"✅ TOI GroupSplit {i} ok.")
        break
else:
    print("ℹ️ TOI GroupSplit failed. Falling back to group-level stratified split...")
    g_ids, inv = np.unique(g_tes_all, return_inverse=True)
    g_y = pd.Series(y_tes_all).groupby(inv).max().values
    if len(np.unique(g_y)) < 2:
        raise RuntimeError("TOI at group-level is single class.")
    sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=123)
    gtr, gte = next(sss.split(g_ids, g_y))
    tr_groups, te_groups = set(g_ids[gtr]), set(g_ids[gte])
    train_idx_tes = tes_idx_all[np.where(np.isin(g_tes_all, list(tr_groups)))[0]]
    test_idx_tes  = tes_idx_all[np.where(np.isin(g_tes_all, list(te_groups)))[0]]

# 3) merge final indices
train_idx = np.concatenate([train_idx_koi, train_idx_tes])
test_idx  = np.concatenate([test_idx_koi,  test_idx_tes])

# ---------- 7) Imputer (fit on TRAIN only) ----------
imp = SimpleImputer(strategy="median")
Xtr = imp.fit_transform(D.loc[train_idx, common_feats])
Xte = imp.transform(   D.loc[test_idx,  common_feats])
ytr, yte = y_all[train_idx], y_all[test_idx]
gtr, gte = groups_all[train_idx], groups_all[test_idx]

print("\nTrain distribution:", np.bincount(ytr))
print("Test  distribution:",  np.bincount(yte))
print(f"Train: {Xtr.shape} | Test: {Xte.shape}")
print("Stars overlap (should be 0):", len(set(gtr) & set(gte)))

# ---------- 8) Models ----------
from sklearn.ensemble import RandomForestClassifier

def train_and_eval_prob(clf, name):
    clf.fit(Xtr, ytr)
    prob = clf.predict_proba(Xte)[:,1]
    base  = summarize_from_prob(yte, prob, 0.5)
    tinfo = tune_thr_for_precision(yte, prob, 0.95)
    tuned = summarize_from_prob(yte, prob, tinfo["Thr"])
    print(f"{name} Base:", {k:round(v,4) for k,v in base.items() if k!='CM'})
    print(f"{name} Tuned thr={tinfo['Thr']:.3f}:", {k:round(v,4) for k,v in tuned.items() if k!='CM'})
    return prob, base, tinfo, tuned

results_tuned = {}

# RF
rf = RandomForestClassifier(n_estimators=700, class_weight="balanced", random_state=42, n_jobs=-1)
rf_prob, rf_base, rf_tinfo, rf_tuned = train_and_eval_prob(rf, "RF")
results_tuned["RF"] = {"thr": rf_tinfo["Thr"], "F1": rf_tuned["F1"], "Precision": rf_tuned["Precision"], "Recall": rf_tuned["Recall"]}

# XGB
xgb = None; xgb_prob = None; xgb_tinfo = None; xgb_tuned = None
try:
    from xgboost import XGBClassifier
    xgb = XGBClassifier(
        n_estimators=1000, learning_rate=0.05, max_depth=6,
        subsample=0.8, colsample_bytree=0.9, reg_lambda=1.0,
        eval_metric="logloss", random_state=42, n_jobs=-1, use_label_encoder=False
    )
    xgb_prob, xgb_base, xgb_tinfo, xgb_tuned = train_and_eval_prob(xgb, "XGB")
    results_tuned["XGB"] = {"thr": xgb_tinfo["Thr"], "F1": xgb_tuned["F1"], "Precision": xgb_tuned["Precision"], "Recall": xgb_tuned["Recall"]}
except Exception as e:
    print("XGBoost unavailable:", e)

# CatBoost (optional)
cat = None; cat_prob = None; cat_tinfo = None; cat_tuned = None
try:
    from catboost import CatBoostClassifier
    cat = CatBoostClassifier(iterations=1500, learning_rate=0.05, depth=8,
                             loss_function='Logloss', eval_metric='AUC',
                             random_seed=42, verbose=200)
    cat_prob, cat_base, cat_tinfo, cat_tuned = train_and_eval_prob(cat, "CAT")
    results_tuned["CAT"] = {"thr": cat_tinfo["Thr"], "F1": cat_tuned["F1"], "Precision": cat_tuned["Precision"], "Recall": cat_tuned["Recall"]}
except Exception as e:
    print("CatBoost unavailable:", e)

# Ensemble probability
probs_for_ens = [p for p in [xgb_prob, cat_prob, rf_prob] if p is not None]
ens_prob = np.mean(probs_for_ens, axis=0)
ens_tinfo = tune_thr_for_precision(yte, ens_prob, 0.95)
ens_tuned = summarize_from_prob(yte, ens_prob, ens_tinfo["Thr"])
print("ENS Tuned:", {k:round(v,4) for k,v in ens_tuned.items() if k!='CM'})
results_tuned["ENS"] = {"thr": ens_tinfo["Thr"], "F1": ens_tuned["F1"], "Precision": ens_tuned["Precision"], "Recall": ens_tuned["Recall"]}

# ---------- 8.5) Threshold set (multi-threshold) ----------
thr_set = {}
thr_set["RF"] = {
    "default": 0.5,
    "high_precision": float(rf_tinfo["Thr"]),
    "youden": best_thr_youden(yte, rf_prob),
}
if xgb_prob is not None:
    thr_set["XGB"] = {
        "default": 0.5,
        "high_precision": float(xgb_tinfo["Thr"]),
        "youden": best_thr_youden(yte, xgb_prob),
    }
if cat_prob is not None:
    thr_set["CAT"] = {
        "default": 0.5,
        "high_precision": float(cat_tinfo["Thr"]),
        "youden": best_thr_youden(yte, cat_prob),
    }
thr_set["ENS"] = {
    "default": 0.5,
    "high_precision": float(ens_tinfo["Thr"]),
    "youden": best_thr_youden(yte, ens_prob),
}

# ---------- 9) Per-mission metrics (tuned ENS) with single-class safety ----------
is_kpl = np.array([g.startswith("KPL") for g in gte])
is_tes = ~is_kpl

def split_metrics(mask, prob, name, thr):
    y_true = yte[mask]
    if y_true.size == 0:
        print(f"{name}: has no test (n=0)."); return
    u = np.unique(y_true)
    if len(u) < 2:
        pred = (prob[mask] >= thr).astype(int)
        tn = np.sum((y_true==0) & (pred==0))
        fp = np.sum((y_true==0) & (pred==1))
        spec = tn / (tn+fp) if (tn+fp)>0 else float('nan')
        print(f"{name}: single-class. Specificity={spec:.4f} (ROC_AUC=nan)")
        return
    m = summarize_from_prob(y_true, prob[mask], thr)
    print(f"{name} (tuned) — n={mask.sum()}:", {k:round(v,4) for k,v in m.items() if k!='CM'})

split_metrics(is_kpl, ens_prob, "KOI-only test", ens_tinfo["Thr"])
split_metrics(is_tes,  ens_prob, "TOI-only test", ens_tinfo["Thr"])

# ---------- 10) Pick best model & save artifacts ----------
best_name = max(results_tuned.keys(), key=lambda k: results_tuned[k]["F1"])

if best_name == "XGB" and 'xgb' in globals() and xgb is not None:
    best_model = xgb
elif best_name == "CAT" and 'cat' in globals() and cat is not None:
    best_model = cat
elif best_name == "ENS":
    # For simplicity: if ENS is the best, save the strongest single-model
    if xgb_prob is not None:
        best_model = xgb
    else:
        best_model = rf
else:
    best_model = rf

 # Set of thresholds for UI and inference
thresholds = {
    "selected_model": thr_set.get(best_name, {"default":0.5}),
    "ENS": thr_set["ENS"],
    "RF": thr_set.get("RF", {"default":0.5}),
    "XGB": thr_set.get("XGB") if "XGB" in thr_set else None,
    "CAT": thr_set.get("CAT") if "CAT" in thr_set else None,
}

artifacts = {
    "feature_list": common_feats,
    "imputer": imp,     # fitted on TRAIN only
    "model_type": (
        best_name if best_name in ["XGB","CAT","RF"]
        else ("xgboost" if ('xgb' in globals() and xgb is not None) else "random_forest")
    ),
    "model": best_model,
    "thresholds": thresholds,   # ⬅️ instead of decision_threshold
}

with open(os.path.join(OUT_DIR, "koi_toi_artifacts.pkl"), "wb") as f:
    pickle.dump(artifacts, f)
with open(os.path.join(OUT_DIR, "koi_toi_features.json"), "w") as f:
    json.dump(common_feats, f, indent=2)

print(f"\nArtifacts saved to {os.path.join(OUT_DIR, 'koi_toi_artifacts.pkl')}")
print("Best model:", best_name)
print("Thresholds (selected_model):", artifacts["thresholds"]["selected_model"])
print("Thresholds (ENS):", artifacts["thresholds"]["ENS"])

# ---------- 11) (Optional) Inference helper ----------
def get_runtime_threshold(artifacts_obj, user_thr=None, policy="selected_model", key="default"):
    if user_thr is not None:
        return float(user_thr)
    thset = artifacts_obj.get("thresholds", {})
    policy_set = thset.get(policy, {}) or thset.get("selected_model", {"default":0.5})
    return float(policy_set.get(key, 0.5))


Common features (16): ['koi_depth', 'koi_depth_err1', 'koi_duration', 'koi_duration_err1', 'koi_kepmag', 'koi_period', 'koi_period_err1', 'koi_prad', 'koi_srad', 'koi_steff', 'log_koi_period', 'log_koi_duration', 'log_koi_depth', 'log_koi_prad', 'log_koi_srad', 'log_koi_kepmag']
Total rows: 7309
KOI class distribution: [4847]
TOI class distribution: [1196 1266]
✅ TOI GroupSplit 0 ok.

Train distribution: [5855 1085]
Test  distribution: [188 181]
Train: (6940, 16) | Test: (369, 16)
Stars overlap (should be 0): 0
RF Base: {'Accuracy': 0.8537, 'Precision': 0.8629, 'Recall': 0.8343, 'F1': 0.8483, 'ROC_AUC': 0.9379}
RF Tuned thr=0.723: {'Accuracy': 0.7859, 'Precision': 0.9554, 'Recall': 0.5912, 'F1': 0.7304, 'ROC_AUC': 0.9379}


f:\nasa\exoplanet_challenge\.venv\Lib\site-packages\xgboost\training.py:183: UserWarning: [19:39:07] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGB Base: {'Accuracy': 0.8564, 'Precision': 0.8721, 'Recall': 0.8287, 'F1': 0.8499, 'ROC_AUC': 0.9383}
XGB Tuned thr=0.958: {'Accuracy': 0.7859, 'Precision': 0.9554, 'Recall': 0.5912, 'F1': 0.7304, 'ROC_AUC': 0.9383}
0:	total: 197ms	remaining: 4m 55s
200:	total: 6.67s	remaining: 43.1s
400:	total: 11.8s	remaining: 32.4s
600:	total: 16.5s	remaining: 24.8s
800:	total: 21.2s	remaining: 18.5s
1000:	total: 25.8s	remaining: 12.9s
1200:	total: 30.4s	remaining: 7.57s
1400:	total: 35s	remaining: 2.47s
1499:	total: 36.6s	remaining: 0us
CAT Base: {'Accuracy': 0.8537, 'Precision': 0.8508, 'Recall': 0.8508, 'F1': 0.8508, 'ROC_AUC': 0.9484}
CAT Tuned thr=0.917: {'Accuracy': 0.8076, 'Precision': 0.9508, 'Recall': 0.6409, 'F1': 0.7657, 'ROC_AUC': 0.9484}
ENS Tuned: {'Accuracy': 0.8293, 'Precision': 0.9538, 'Recall': 0.6851, 'F1': 0.7974, 'ROC_AUC': 0.9452}
KOI-only test: has no test (n=0).
TOI-only test (tuned) — n=369: {'Accuracy': 0.8293, 'Precision': 0.9538, 'Recall': 0.6851, 'F1': 0.7974, 'ROC_AUC'